# Task 6 - Results from the current experiments

The summary reads JSON generated by Tasks 2-5. No metric or training time is filled in
manually. Missing experiments remain absent. Smoke and full runs, datasets, evaluation
stages and training protocols are labelled separately.

Classical models are selected using validation F1 and refitted on train + validation.
The deep-learning primary models use training data plus validation for early stopping.
Their test scores therefore have different training protocols and are not a controlled
head-to-head comparison.

## 1. Deep Learning Algorithm Survey (600–800 words)

The landscape of deep learning has expanded dramatically beyond the feedforward networks of Tasks 4–5. This survey covers five transformative architectures that have shaped modern AI.

---

### 1.1 Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM)

**Purpose:** RNNs are designed for sequential data where temporal order matters. Unlike feedforward networks, RNNs maintain a hidden state `hₜ = f(Whₜ₋₁ + Uxₜ + b)` that carries information across time steps, enabling the network to model dependencies in sequences.

**Key Innovation:** LSTMs (Hochreiter & Schmidhuber, 1997) solve the vanishing gradient problem in vanilla RNNs through a gated memory cell. The three gates — input gate `i`, forget gate `f`, and output gate `o` — selectively store, delete, and output information:
```
fₜ = σ(Wf·[hₜ₋₁, xₜ] + bf)    # Forget gate
iₜ = σ(Wi·[hₜ₋₁, xₜ] + bi)    # Input gate  
cₜ = fₜ⊙cₜ₋₁ + iₜ⊙tanh(Wc·[hₜ₋₁, xₜ])  # Cell state update
```
The cell state `cₜ` can carry information across hundreds of time steps unchanged.

**Real-world application:** Speech recognition (Google's voice search), machine translation, time-series forecasting (stock prices, weather prediction), and text generation.

---

### 1.2 Generative Adversarial Networks (GANs)

**Purpose:** GANs (Goodfellow et al., 2014) learn to generate realistic data samples from a learned distribution, enabling synthesis of images, audio, and text that are indistinguishable from real data.

**Key Innovation:** The adversarial training framework pits two networks against each other. A Generator `G(z)` maps noise vectors `z ~ p(z)` to fake data, while a Discriminator `D(x)` classifies inputs as real or fake. The minimax objective is:
```
min_G max_D E[log D(x)] + E[log(1 - D(G(z)))]
```
This adversarial dynamic drives both networks to improve until the generator produces samples indistinguishable from real data.

**Real-world application:** Photorealistic image synthesis (StyleGAN3), medical image augmentation for rare diseases, deepfake detection (via discriminator), drug molecule generation in computational biology.

---

### 1.3 Transformers and the Attention Mechanism

**Purpose:** Transformers (Vaswani et al., 2017) process sequences in parallel (unlike RNNs) by relating every position to every other position via attention, enabling unprecedented scaling in NLP and beyond.

**Key Innovation:** Multi-Head Self-Attention computes queries Q, keys K, and values V from the input, then:
```
Attention(Q,K,V) = softmax(QKᵀ / √dₖ) · V
```
The `√dₖ` scaling prevents dot products from becoming too large in high dimensions. Multiple heads capture different relationship types. Positional encodings inject sequence order since attention itself is permutation-invariant.

**Real-world application:** GPT-4 and Claude for language generation, BERT for search and Q&A, Vision Transformers (ViT) for image classification, AlphaFold2 for protein structure prediction.

---

### 1.4 Graph Neural Networks (GNNs)

**Purpose:** GNNs extend deep learning to graph-structured data where relationships between entities are as important as the entities themselves — social networks, molecular graphs, knowledge bases, and citation networks.

**Key Innovation:** Message passing aggregates information from a node's neighborhood:
```
h_v^(k) = UPDATE(h_v^(k-1), AGGREGATE({h_u^(k-1) : u ∈ N(v)}))
```
Graph Convolutional Networks (Kipf & Welling, 2017) implement this as:
`H^(k) = σ(D̃^(-1/2) Ã D̃^(-1/2) H^(k-1) W^(k))`
where `Ã = A + I` is the adjacency matrix with self-loops. This allows information to propagate through the graph over k layers.

**Real-world application:** Drug discovery (molecular property prediction), fraud detection in financial transaction graphs, recommendation systems (Pinterest), traffic prediction (Google Maps).

---

### 1.5 Autoencoders and Variational Autoencoders (VAEs)

**Purpose:** Autoencoders learn compressed, low-dimensional latent representations by encoding input data into a bottleneck and reconstructing it. VAEs extend this to probabilistic generative models.

**Key Innovation (VAE):** Instead of encoding to a point, VAEs encode to a distribution `q(z|x) = N(μ, σ²)`. The ELBO (Evidence Lower BOund) objective:
```
L = E[log p(x|z)] - KL(q(z|x) || p(z))
```
The KL term regularizes the latent space toward a standard normal, enabling smooth interpolation and generation. The reparameterization trick `z = μ + σ·ε, ε~N(0,I)` makes the sampling step differentiable.

**Real-world application:** Anomaly detection (VAE reconstruction error flags outliers), drug discovery (generating novel molecular structures), image compression, and as a backbone for conditional generation (DALL-E v1).

---

In [ ]:
from pathlib import Path
import pandas as pd
from ml_lab.reporting import collect_reports

paths = sorted(Path("results").glob("task[2345]_*.json"))
if not paths:
    raise FileNotFoundError("No generated reports. Run Task 2, 3, 4 or 5 first.")
rows = collect_reports(paths)
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_csv("results/current_summary.csv", index=False)
print("Missing reports are omitted; no zero scores or historical defaults are substituted.")


## Read the evidence

Filter by dataset, mode, protocol, stage and metric before making a comparison.
CV scores used for tuning can be optimistic after hyperparameter selection.
A final held-out result describes this particular data split and protocol, not a
universal model ranking. Training time depends on hardware and was not measured
consistently across the original experiments, so this notebook does not rank it.

The old manually populated result CSVs have been removed from the current version.
Previous notebook outputs were cleared after fixing leakage; rerun to obtain new results.

The Diabetes example is educational, is limited to its source population and requires
separate validation for any clinical use. This project does not provide medical advice
or a deployed diagnostic model.